# Fine-Tune OpenLID via PyTorch Head Expansion
This notebook implements a novel incremental learning methodology for FastText models. 
Since FastText `.bin` models cannot dynamically expand their internal vocabulary/label dictionaries out-of-the-box, we use the pre-trained OpenLID model as a frozen feature extractor and migrate its classification head to PyTorch. This allows us to mathematically expand the output matrix to support new languages without catastrophic forgetting (when combined with a replay buffer).

In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas numpy torch huggingface_hub')
    # fasttext wheel can sometimes be tricky on colab, so we compile it
    os.system('pip install -q fasttext')
    print("Setup complete!")


## Step 1: Download and Setup the Base OpenLID Model
We use the `HPLT/OpenLID-v3` model from Hugging Face. This model is compact and perfectly suited for this expansion architecture because it was trained using standard softmax, making its output matrix mathematically sound for 1:1 migration.

In [2]:
import os
from huggingface_hub import hf_hub_download
import fasttext

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

model_dir = 'models/pretrained/openlid'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'openlid-v3.bin')

if not os.path.exists(model_path):
    print("Downloading OpenLID-v3 from HuggingFace...")
    downloaded_path = hf_hub_download(repo_id="HPLT/OpenLID-v3", filename="openlid-v3.bin")
    # Symlink or move it to our local models dir
    os.system(f'cp {downloaded_path} {model_path}')
    print(f"Model downloaded and saved to {model_path}")
else:
    print(f"Model already exists at {model_path}")

print("Loading FastText model (this may take a moment)...")
# Suppress fasttext warning
fasttext.FastText.eprint = lambda x: None
ft_model = fasttext.load_model(model_path)
print("OpenLID model loaded successfully!")

old_labels = ft_model.get_labels()
hidden_dim = ft_model.get_dimension()
print(f"Original Model supports {len(old_labels)} languages.")
print(f"Hidden Dimension: {hidden_dim}")


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model downloaded and saved to models/pretrained/openlid/openlid-v3.bin
Loading FastText model (this may take a moment)...
OpenLID model loaded successfully!
Original Model supports 195 languages.
Hidden Dimension: 256


In [3]:
old_labels = ft_model.get_labels()

print(f"Total Labels: {len(old_labels)}")
print("All labels:")
print(old_labels)
print("=============================================================")

# Check if specific languages are already in the model
# FastText usually prepends "__label__" to the classes
languages_to_check = ["__label__sin", "__label__san", "__label__pli", "__label__eng"]

print("\nChecking for specific languages:")
for lang in languages_to_check:
    if lang in old_labels:
        print(f"✅ {lang} is natively supported at index {old_labels.index(lang)}")
    else:
        print(f"❌ {lang} is MISSING from the original model.")


Total Labels: 195
All labels:
['__label__rus_Cyrl', '__label__eng_Latn', '__label__ara_Arab', '__label__por_Latn', '__label__dan_Latn', '__label__pol_Latn', '__label__ekk_Latn', '__label__ell_Grek', '__label__fas_Arab', '__label__slk_Latn', '__label__slv_Latn', '__label__nld_Latn', '__label__hun_Latn', '__label__lvs_Latn', '__label__swe_Latn', '__label__lit_Latn', '__label__fin_Latn', '__label__heb_Hebr', '__label__cmn_Hant', '__label__mlt_Latn', '__label__cat_Latn', '__label__nob_Latn', '__label__ind_Latn', '__label__tam_Taml', '__label__ben_Beng', '__label__zxx_Zxxx', '__label__uzn_Latn', '__label__kin_Latn', '__label__fil_Latn', '__label__ukr_Cyrl', '__label__hin_Deva', '__label__ast_Latn', '__label__cmn_Hans', '__label__afr_Latn', '__label__mar_Deva', '__label__ceb_Latn', '__label__ilo_Latn', '__label__zul_Latn', '__label__xho_Latn', '__label__jpn_Jpan', '__label__vie_Latn', '__label__guj_Gujr', '__label__amh_Ethi', '__label__hrv_Latn', '__label__nya_Latn', '__label__tsn_Latn', '__

In [4]:
for label in old_labels:
    if 'sin' in label:
        print("Sin label - ", label)
    elif 'pli' in label or 'pali' in label:
        print("Pali label - ", label)
    elif 'san' in label:
        print("San label - ", label)

Sin label -  __label__sin_Sinh
San label -  __label__san_Deva


In [5]:
W_old = ft_model.get_output_matrix()

print(f"Output Matrix Shape: {W_old.shape}")
print(f"Number of rows (labels): {W_old.shape[0]}")
print(f"Number of columns (hidden embedding dimension): {W_old.shape[1]}")

# You can even look at the raw weights for a specific language
eng_index = old_labels.index("__label__eng") if "__label__eng" in old_labels else 0
print(f"\nFirst 5 weights in the English classification node:\n{W_old[eng_index][:5]}")


Output Matrix Shape: (195, 256)
Number of rows (labels): 195
Number of columns (hidden embedding dimension): 256

First 5 weights in the English classification node:
[ 3.6524394   2.220934    2.2925477   0.19384927 -1.082639  ]


In [6]:
sample_text = "This is a test sentence in English."

# get_sentence_vector() automatically hashes the character n-grams 
# and averages them into a single fixed-size vector.
embedding = ft_model.get_sentence_vector(sample_text)

print(f"Embedding Shape: {embedding.shape}")
print(f"First 10 values of the embedding:\n{embedding[:10]}")


Embedding Shape: (256,)
First 10 values of the embedding:
[ 0.04606908 -0.02452674  0.04748058 -0.05892856  0.08173286 -0.02769783
  0.13285641  0.10186785 -0.00173775  0.05462543]


In [7]:
words = ft_model.get_words()
print(f"Total vocabulary size (words + subwords): {len(words)}")

print("\nSample of stored subwords/tokens:")
print(words[100:110]) 


Total vocabulary size (words + subwords): 199731

Sample of stored subwords/tokens:
['të', 'l', 'til', 'det', '့', 'за', 'em', 'mi', 'der', 'om']


## Step 2: Extract Weights & Expand the PyTorch Head
Here we extract the existing `W_old` weights, append random initialized weights for our new languages, and inject them into a PyTorch `nn.Linear` layer without biases.

In [8]:
import torch
import torch.nn as nn
import numpy as np

print("Extracting FastText weights...")
old_labels = ft_model.get_labels()
W_old = ft_model.get_output_matrix()
hidden_dim = ft_model.get_dimension()

# Mapping our dataset 3-letter codes to OpenLID's ISO3_Script format
DATASET_TO_OPENLID_MAP = {
    "eng": "__label__eng_Latn",
    "sin": "__label__sin_Sinh",
    "san": "__label__san_Sinh", # We are adding Sinhala-script Sanskrit
    "pli": "__label__pli_Sinh", # We are adding Sinhala-script Pali
    "tam": "__label__tam_Taml",
    "hin": "__label__hin_Deva",
    "ben": "__label__ben_Beng",
    "arb": "__label__ara_Arab", # Notice OpenLID uses 'ara' instead of 'arb'
    "fra": "__label__fra_Latn",
    "deu": "__label__deu_Latn",
    "jpn": "__label__jpn_Jpan",
    "nld": "__label__nld_Latn",
    "pol": "__label__pol_Latn",
    "ita": "__label__ita_Latn",
    "por": "__label__por_Latn",
    "tur": "__label__tur_Latn",
    "spa": "__label__spa_Latn",
    "ell": "__label__ell_Grek",
    "urd": "__label__urd_Arab",
    "bul": "__label__bul_Cyrl",
    "cmn": "__label__cmn_Hans",
    "rus": "__label__rus_Cyrl",
    "tha": "__label__tha_Thai",
    "swh": "__label__swh_Latn",
    "vie": "__label__vie_Latn"
}

# Find which labels need to be mathematically ADDED to the matrix
new_labels = []
for target_label in DATASET_TO_OPENLID_MAP.values():
    if target_label not in old_labels:
        new_labels.append(target_label)

print(f"\nFound {len(new_labels)} labels completely missing from the model. Adding them to expansion:")
for label in new_labels:
    print(f"  -> {label}")

num_new_labels = len(new_labels)

print("\nExpanding PyTorch classification head...")
# Create random weights for the new labels
W_new_init = np.random.normal(scale=0.1, size=(num_new_labels, hidden_dim))
W_combined = np.vstack([W_old, W_new_init])

# Create the extended PyTorch head (no bias, just like FastText)
extended_head = nn.Linear(hidden_dim, len(old_labels) + num_new_labels, bias=False)
extended_head.weight.data = torch.tensor(W_combined, dtype=torch.float32)
extended_head.train() # Set to train mode

print("PyTorch Head created successfully!")
print(f"New Head Shape: {extended_head.weight.shape}")


Extracting FastText weights...

Found 2 labels completely missing from the model. Adding them to expansion:
  -> __label__san_Sinh
  -> __label__pli_Sinh

Expanding PyTorch classification head...
PyTorch Head created successfully!
New Head Shape: torch.Size([197, 256])


## Step 3: PyTorch Training Loop
We create a custom PyTorch Dataset. For every sentence, we use the **frozen** FastText model to generate the 256-dimensional embedding `get_sentence_vector()`, and pass that embedding to our PyTorch head for loss calculation and backprop.

In [9]:
import pandas as pd
import json
import copy
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print("Defining PyTorch Dataset class...")
class FastTextDataset(Dataset):
    def __init__(self, jsonl_path, ft_model, label2id, dataset_map):
        self.ft_model = ft_model
        self.label2id = label2id
        self.dataset_map = dataset_map
        
        records = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                records.append(json.loads(line))
        self.df = pd.DataFrame(records)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['text']).replace('\n', ' ') 
        raw_label = self.df.iloc[idx]['label']
        
        ft_label = self.dataset_map.get(raw_label)
        target_idx = self.label2id.get(ft_label, 0)
            
        features = self.ft_model.get_sentence_vector(text)
        return torch.tensor(features, dtype=torch.float32), torch.tensor(target_idx, dtype=torch.long)

def build_fresh_head():
    """Creates a fresh PyTorch head using the original FastText weights"""
    head = nn.Linear(hidden_dim, len(old_labels) + num_new_labels, bias=False)
    head.weight.data = torch.tensor(W_combined, dtype=torch.float32)
    return head


Defining PyTorch Dataset class...


In [10]:
def train_experiment(train_path, val_path, save_name, epochs=10, patience=2):
    print(f"\n{'='*50}")
    print(f"STARTING EXPERIMENT: {save_name}")
    print(f"Train Data: {train_path}")
    print(f"Val Data: {val_path}")
    print(f"{'='*50}")
    
    all_labels = old_labels + new_labels
    label2id = {label: i for i, label in enumerate(all_labels)}
    
    train_dataset = FastTextDataset(train_path, ft_model, label2id, DATASET_TO_OPENLID_MAP)
    val_dataset = FastTextDataset(val_path, ft_model, label2id, DATASET_TO_OPENLID_MAP)
    
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    head = build_fresh_head().to(device)
    
    optimizer = AdamW(head.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_weights = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        # Training Loop
        head.train()
        train_loss = 0
        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(head(features), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation Loop
        head.eval()
        val_loss = 0
        with torch.no_grad():
            for features, targets in val_loader:
                features, targets = features.to(device), targets.to(device)
                loss = criterion(head(features), targets)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        # Early Stopping Logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = copy.deepcopy(head.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s).")
            if epochs_no_improve >= patience:
                print("Early stopping triggered!")
                break
                
    # Save Best Model
    save_dir = "models/finetuned/openlid"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{save_name}.pt")
    torch.save(best_weights, save_path)
    print(f"Experiment complete. Best model saved to {save_path}")
    return save_path


## Experiment 1: No Rehearsal (To measure Catastrophic Forgetting)
We train exclusively on the Sinhala-script dataset. This should violently overwrite the existing representations.

In [11]:
train_experiment(
    train_path='datasets/finetuning/train.jsonl',
    val_path='datasets/finetuning/val_mixed.jsonl',
    save_name='openlid_head_no_rehearsal',
    epochs=10,
    patience=2
)



STARTING EXPERIMENT: openlid_head_no_rehearsal
Train Data: datasets/finetuning/train.jsonl
Val Data: datasets/finetuning/val_mixed.jsonl
Epoch 1/10 | Train Loss: 10.7075 | Val Loss: 7.7046
Epoch 2/10 | Train Loss: 8.3122 | Val Loss: 5.7712
Epoch 3/10 | Train Loss: 5.9281 | Val Loss: 3.8464
Epoch 4/10 | Train Loss: 3.5771 | Val Loss: 2.0011
Epoch 5/10 | Train Loss: 1.6156 | Val Loss: 0.8480
Epoch 6/10 | Train Loss: 0.8458 | Val Loss: 0.6459
Epoch 7/10 | Train Loss: 0.7498 | Val Loss: 0.6164
Epoch 8/10 | Train Loss: 0.7212 | Val Loss: 0.5953
Epoch 9/10 | Train Loss: 0.6979 | Val Loss: 0.5767
Epoch 10/10 | Train Loss: 0.6769 | Val Loss: 0.5594
Experiment complete. Best model saved to models/finetuned/openlid/openlid_head_no_rehearsal.pt


'models/finetuned/openlid/openlid_head_no_rehearsal.pt'

## Experiment 2: With Rehearsal (The Actual Research Solution)
We train on the mixed dataset (Sinhala scripts + Aya dataset subset). This forces the model to balance learning the new target languages while remembering the original base languages.

In [12]:
train_experiment(
    train_path='datasets/finetuning/train_mixed.jsonl',
    val_path='datasets/finetuning/val_mixed.jsonl',
    save_name='openlid_head_with_rehearsal',
    epochs=10,
    patience=2
)



STARTING EXPERIMENT: openlid_head_with_rehearsal
Train Data: datasets/finetuning/train_mixed.jsonl
Val Data: datasets/finetuning/val_mixed.jsonl
Epoch 1/10 | Train Loss: 6.2421 | Val Loss: 6.0461
Epoch 2/10 | Train Loss: 3.6013 | Val Loss: 2.5227
Epoch 3/10 | Train Loss: 1.3984 | Val Loss: 0.7271
Epoch 4/10 | Train Loss: 0.7545 | Val Loss: 0.6722
Epoch 5/10 | Train Loss: 0.5775 | Val Loss: 0.6353
Epoch 6/10 | Train Loss: 0.4837 | Val Loss: 0.5930
Epoch 7/10 | Train Loss: 0.4318 | Val Loss: 0.5501
Epoch 8/10 | Train Loss: 0.3951 | Val Loss: 0.5077
Epoch 9/10 | Train Loss: 0.3654 | Val Loss: 0.4709
Epoch 10/10 | Train Loss: 0.3424 | Val Loss: 0.4434
Experiment complete. Best model saved to models/finetuned/openlid/openlid_head_with_rehearsal.pt


'models/finetuned/openlid/openlid_head_with_rehearsal.pt'